In [19]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
document_path = "C:\\Users\\Yanovich\\Downloads\\Эффективный конспект по математической статистике_ от корреляции до сложных диагностических процедур.pdf"

In [21]:
loader = PyMuPDF4LLMLoader(document_path)
documents = loader.load()

In [22]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)

In [23]:
print(f"✅ Загружено {len(documents)} страниц, создано {len(chunks)} чанков.")

✅ Загружено 24 страниц, создано 85 чанков.


In [24]:
from langchain_huggingface import HuggingFaceEmbeddings

In [25]:
embeddings = HuggingFaceEmbeddings(model_name="cointegrated/rubert-tiny2")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
from langchain_chroma import Chroma

In [27]:
chromadb_dir = "../chroma_db"
chromadb_collection_name = "slm_answer_checker_rag"

In [28]:
import chromadb

client = chromadb.PersistentClient(path=chromadb_dir)
try:
    client.delete_collection(chromadb_collection_name)
except:
    pass

In [29]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=chromadb_collection_name,
    persist_directory=chromadb_dir
)

In [30]:
from langchain_ollama import ChatOllama
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

In [31]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOllama(model="qwen3:8b")

In [32]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_classic.prompts import PromptTemplate

In [33]:
class AnswerSchema(BaseModel):
    success: bool = Field(description="Успешно ли найден и сгенерирован ответ")
    message: str = Field(description="Содержание ответа на вопрос")

parser = PydanticOutputParser(pydantic_object=AnswerSchema)

prompt_template = """
Ты — ассистент, отвечающий на вопросы по документу. 
Используй следующий контекст для ответа. Если контекст не содержит информации, 
сообщи об этом в поле message и установи success = false.

Контекст:
{context}

Вопрос: {question}

Ответь в формате JSON.
{format_instructions}
"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)

raw_output = qa_chain.invoke("Опишите суть метода наименьших квадратов")

In [34]:
parsed = parser.parse(raw_output["result"])

In [35]:
raw_output

{'query': 'Опишите суть метода наименьших квадратов',
 'result': '{\n  "success": true,\n  "message": "Метод наименьших квадратов (МНК) — это стандартный подход для оценки параметров регрессионной модели. Его суть заключается в минимизации суммы квадратов остатков (разниц между наблюдаемыми и предсказанными моделью значениями). Цель — найти такие значения коэффициентов β₀ и β₁, при которых общая \'энергия\' ошибок будет минимальной. Это приводит к системе уравнений, решение которой определяет формулы для коэффициентов: наклон прямой (β₁) рассчитывается как отношение ковариации между x и y к дисперсии x, а свободный член (β₀) выбирается так, чтобы линия проходила через точку средних значений (x̄, ȳ)."\n}',
 'source_documents': [Document(id='960436f7-bc22-4b02-9a62-3d6070992ae0', metadata={'subject': '', 'page': 3, 'total_pages': 24, 'format': 'PDF 1.7', 'keywords': '', 'file_path': 'C:\\Users\\Yanovich\\Downloads\\Эффективный конспект по математической статистике_ от корреляции до сложн

In [36]:
parsed

AnswerSchema(success=True, message="Метод наименьших квадратов (МНК) — это стандартный подход для оценки параметров регрессионной модели. Его суть заключается в минимизации суммы квадратов остатков (разниц между наблюдаемыми и предсказанными моделью значениями). Цель — найти такие значения коэффициентов β₀ и β₁, при которых общая 'энергия' ошибок будет минимальной. Это приводит к системе уравнений, решение которой определяет формулы для коэффициентов: наклон прямой (β₁) рассчитывается как отношение ковариации между x и y к дисперсии x, а свободный член (β₀) выбирается так, чтобы линия проходила через точку средних значений (x̄, ȳ).")